In [ ]:
import json 
import pandas as pd 

df = pd.read_json('data/part50/Action_json_part50.json')

df.head()



In [ ]:
df.info()

In [ ]:
df_nlp = df[['review', 'voted_up','votes_up','votes_funny', 'author.playtime_at_review', 'language']]

df_nlp.head()

In [ ]:
# Cria uma coluna nova dividindo os minutos por 60
df['horas_jogadas_na_review'] = (df['author.playtime_at_review'] / 60).round(2)

In [ ]:
import pandas as pd

# TRUQUE DE MESTRE: Por padrão, o Pandas corta textos longos com "..."
# Essa linha abaixo desativa esse limite, permitindo ler a review completa.
pd.set_option('display.max_colwidth', None)

colunas = ['review', 'voted_up', 'votes_up', 'votes_funny', 'author.playtime_at_review','horas_jogadas_na_review']

df_amostra = df[colunas]

df_amostra.sample(10)

### 1. Amostragem e Concatenação dos Dados
Vamos extrair uma amostra de `N` reviews de cada arquivo JSON para montar um "mini-arquivão" balanceado, sem estourar a memória.

In [ ]:
import os
import pandas as pd

# Caminhos para as pastas
data_dirs = ['data/part50', 'data/rest_part50']
amostras_por_genero = 2000  # Ajuste conforme precisar
df_list = []

print("Iniciando amostragem...")
for directory in data_dirs:
    if os.path.exists(directory):
        for filename in os.listdir(directory):
            if filename.endswith('.json'):
                filepath = os.path.join(directory, filename)
                genero = filename.split('_')[0]
                
                print(f"Lendo e amostrando: {genero} da pasta {directory.split('/')[-1]}...")
                
                try:
                    df_temp = pd.read_json(filepath)
                    
                    # Previne erro se o dataset for menor que a amostra desejada
                    n_amostras = min(amostras_por_genero, len(df_temp))
                    df_sampled = df_temp.sample(n_amostras, random_state=42).copy()
                    
                    # Registra a origem do gênero
                    df_sampled['genre'] = genero 
                    
                    # Colunas vitais para a pipeline ML do Golden Reviews
                    colunas_uteis = [
                        'review', 'voted_up', 'votes_up', 'votes_funny', 
                        'author.playtime_at_review', 'language', 'genre'
                    ]
                    
                    # Garante que só pegaremos colunas que existem no dataframe atual
                    colunas_existentes = [col for col in colunas_uteis if col in df_sampled.columns]
                    
                    df_list.append(df_sampled[colunas_existentes])
                except Exception as e:
                    print(f"Erro ao processar {filename}: {e}")

# Concatena todos os pedaços num dataframe único
df_corpus = pd.concat(df_list, ignore_index=True)

print(f"\nAmostragem concluída!")
print(f"Tamanho total do Corpus Balanceado: {df_corpus.shape}")

# Vizualização das primeiras linhas e resumo da distribuição
display(df_corpus.head())
df_corpus['genre'].value_counts()

### 2. Limpeza de Dados Segura (Safe Data Cleaning)
Modelos de linguagem modernos (como DistilBERT) dependem de contexto, pontuação e estrutura da frase. Abordagens antigas como "arrancar todas as Stop Words", "lematizar" ou "deixar tudo em minúsculo" destroem a semântica e **pioram** a performance desses modelos.

Nossa limpeza focará apenas em "Data Quality":
- Apenas avaliações em Português/BR.
- Reviews de jogadores com mais de 30 minutos jogados (evitar opiniões vazias).
- Remoção de ASCII Art (desenhos de caracteres comuns na Steam) e URLs.
- Eliminação de frases com menos de 3 palavras (ex: "Top", "Legal").
- Preservação rigorosa de pontuações vitais (`.`, `!`, `?`), pois usaremos elas na próxima etapa para quebrar as reviews em frases.

In [ ]:
import re

# 1. Copiando o corpus original
df_clean = df_corpus.copy()

# 2. Filtro de Idioma e Tempo de Jogo (Mínimo 30 minutos)
df_clean = df_clean[df_clean['language'].isin(['portuguese', 'brazilian'])]
df_clean = df_clean[df_clean['author.playtime_at_review'] >= 30]

# 3. Função de Limpeza Segura
def clean_text_safe(text):
    if not isinstance(text, str):
        return ""
    
    # Remove ASCII Art comum na Steam (ex: caracteres Braille usados para desenhar)
    text = re.sub(r'[\u2800-\u28FF]', '', text)
    
    # Remove links
    text = re.sub(r'http\S+', '', text)
    
    # Substitui quebras de linha por um ponto ou espaço (ajuda na segmentação depois)
    text = re.sub(r'\n+', '. ', text)
    
    # Corrige múltiplos espaços repetidos
    text = re.sub(r'\s+', ' ', text)
    
    # Remove pontuações repetidas que causam ruído ao modelo (ex: !!!!! -> !)
    text = re.sub(r'([.!?])\1+', r'\1', text)
    
    # Remove caracteres especiais soltos demais (mantém letras, números e pontuação comum)
    text = re.sub(r'[^a-zA-Z0-9À-ÿ.,!?\'\"() ]', '', text)
    
    return text.strip()

print("Executando limpeza de texto. Isso pode levar alguns segundos...")
df_clean['review_clean'] = df_clean['review'].apply(clean_text_safe)

# 4. Filtro de Tamanho (Excluir reviews super curtas com menos de 3 palavras)
df_clean['word_count'] = df_clean['review_clean'].str.split().str.len()
df_clean = df_clean[df_clean['word_count'] >= 3]

print(f"Tamanho antes da limpeza: {df_corpus.shape[0]}")
print(f"Tamanho após a limpeza: {df_clean.shape[0]}")
print(f"Total de reviews descartadas (ruído): {df_corpus.shape[0] - df_clean.shape[0]}\n")

# Mostrando um antes e depois
display(df_clean[['review', 'review_clean']].sample(5))

### 3. Segmentação de Sentenças (Sentence Segmentation)
Aqui entra o "Pulo do Gato" para o modelo Híbrido: quebrar paredes de texto em frases individuais. 
Isso permite que um LLM julgue `"O gráfico é lindo." (Positivo)` e `"Mas a história é péssima." (Negativo)` separadamente.

In [ ]:
# Instala e importa o NLTK caso não esteja no ambiente
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Garantia de compatibilidade com versões mais novas do nltk

from nltk.tokenize import sent_tokenize

# Lista para armazenar as novas linhas
linhas_expandidas = []

print("Segmentando as avaliações em frases individuais...")

# Itera pelo dataframe limpo
for index, row in df_clean.iterrows():
    texto = row['review_clean']
    
    # O sent_tokenize já sabe onde cortar (pontos, exclamações, interrogações)
    frases = sent_tokenize(texto, language='portuguese')
    
    # Para cada frase extraída, cria uma nova "linha" herdando os dados originais (gênero, votos, etc)
    for frase in frases:
        if len(frase.split()) >= 3: # Ignora frases residuais muito curtas
            nova_linha = row.copy()
            nova_linha['sentence'] = frase
            linhas_expandidas.append(nova_linha)

# Cria o Dataframe final pronto para o Modelo BERT
df_sentences = pd.DataFrame(linhas_expandidas)
df_sentences.reset_index(drop=True, inplace=True)

print(f"Total de avaliações inteiras originais: {len(df_clean)}")
print(f"Total de frases individuais isoladas: {len(df_sentences)}\n")

display(df_sentences[['sentence', 'genre']].head(10))

### 4. Salvando o Dataset Processado
Para não perdermos esses 190.000 registros e não precisarmos rodar tudo do zero, vamos salvar o resultado.
Salvaremos em **Parquet** (melhor formato para Pandas/ML, mais rápido e leve) e **CSV** (backup universal).

In [ ]:
# Trocando o motor do parquet para 'fastparquet' para desviar de um bug na lib pyarrow
!pip install fastparquet

import os

# Cria uma pasta para os dados processados caso não exista
os.makedirs('data/processed', exist_ok=True)

print("Salvando os dados...")

# 1. Formato Parquet (RECOMENDADO para Machine Learning)
# Usando a engine fastparquet que está mais estável na versão atual
df_sentences.to_parquet('data/processed/steam_reviews_sentences.parquet', engine='fastparquet', index=False)
print("Salvo: data/processed/steam_reviews_sentences.parquet")

# 2. Formato CSV (Backup Universal)
df_sentences.to_csv('data/processed/steam_reviews_sentences.csv', index=False, encoding='utf-8')
print("Salvo: data/processed/steam_reviews_sentences.csv")

print("\nTudo salvo com sucesso! O próximo passo será carregar o modelo DistilBERT do HuggingFace nas nossas frases.")

In [4]:
import pandas as pd

# Carregando o Parquet gerado para validar os acentos
df_check = pd.read_parquet('data/processed/steam_reviews_sentences.parquet', engine='fastparquet')

# O Jupyter trata o encoding utf-8 perfeitamente na interface
print("--- Teste de Caracteres Especiais ---\n")
for i, frase in enumerate(df_check['sentence'].head()):
    print(f"Frase {i+1}: {frase}\n")


--- Teste de Caracteres Especiais ---

Frase 1: Em minha opnião, o melhor postapocalyptic da atualidade, mapa grande ( nao comparado ao chernarus de DayZ ), sistema de bases bom porém com algumas falhas, graficos bonitos, bugs que surgem com as atualizações vão sendo corrigidos com patches menores, gráficos muito bonitos e bem trabalhados, armas fortes com uma certa raridade para serem encontradas e o melhor de tudo, existem 2 servidores Brasileiros oficiais.

Frase 2: Joguei mais um bocado do Evolve, e até terminei a "campanha" single player dele.

Frase 3: Primeira coisa, é que é um jogo muito, muito mesmo bagunçado, sério, várias vezes me senti perdido, sem saber o que estava acontecendo direito na tela, quase um Ultimate Marvel vs Capcom 3 no nível de bagunça na tela.

Frase 4: Então se prepare para ficar puto jogando esse jogo, porque várias vezes você simplesmente vai ter um monstro gigante pulando na sua cabeça, e você nem sabe de onde o desgraçado veio.

Frase 5: Segunda coisa,

In [5]:
# Instalação das bibliotecas de Deep Learning
!pip install torch torchvision torchaudio
!pip install transformers

from transformers import pipeline

# Usando o modelo pysentimiento/bertweet-pt-sentiment focado em PT-BR informal
# Treinado com a base BERTabaporu (tweets brasileiros)
print("Baixando e carregando o modelo de Sentimento... (isso ocorre apenas na 1ª vez)")
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="pysentimiento/bertweet-pt-sentiment"
)

# Vamos testar se o modelo entende o contexto (sem usar palavras-chave!)
frases_teste = [
    "Os gráficos são lindos, mas a jogabilidade é travada e cheia de bugs.",
    "Nunca me diverti tanto em um jogo!",
    "Mais ou menos, não vale o preço atual.",
    "bão demais da conta sô, recomendo a todos"
]

print("\n--- Teste de Inteligência do Modelo ---")
for frase in frases_teste:
    resultado = sentiment_pipeline(frase)[0]
    print(f"Frase: '{frase}'\nClassificação: {resultado['label']} (Confiança: {resultado['score']:.2f})\n")



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


d:\workspace\ludo-prism\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Baixando e carregando o modelo de Sentimento... (isso ocorre apenas na 1ª vez)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6604.53it/s]



--- Teste de Inteligência do Modelo ---
Frase: 'Os gráficos são lindos, mas a jogabilidade é travada e cheia de bugs.'
Classificação: NEG (Confiança: 0.97)

Frase: 'Nunca me diverti tanto em um jogo!'
Classificação: POS (Confiança: 0.98)

Frase: 'Mais ou menos, não vale o preço atual.'
Classificação: NEG (Confiança: 0.65)

Frase: 'bão demais da conta sô, recomendo a todos'
Classificação: POS (Confiança: 0.99)



In [6]:
# Um agradecimento especial ao João Lenda por achar nosso modelo perfeito! 🏆

# O modelo avisou que precisa da biblioteca "emoji" para traduzir carinhas (ex: 😭, 😍).
# Na Steam isso é super comum, então vamos deixar ele ainda mais inteligente.
!pip install emoji==0.6.0

# Um testezinho rápido de como os emojis viram texto por debaixo dos panos:
import emoji

frase_gamer = "Mano o jogo é top demais, zerei em 10 horas 🔥😭"
# A versão 0.6.0 do emoji (que o modelo pede) faz a tradução padrão para o inglês
frase_traduzida = emoji.demojize(frase_gamer)

print(f"Original: {frase_gamer}")
print(f"Como o modelo vai ler: {frase_traduzida}")

Original: Mano o jogo é top demais, zerei em 10 horas 🔥😭
Como o modelo vai ler: Mano o jogo é top demais, zerei em 10 horas :fire::loudly_crying_face:



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Testando em um lote menor para não travar o PC
!pip install tqdm
# Usando o tqdm padrão em vez do notebook para evitar bugs de interface (ipywidgets)
from tqdm import tqdm
tqdm.pandas() # Ativa a barra de progresso (progress_apply) no pandas

# Pegando 500 sentenças aleatórias do nosso parquet carregado (df_check)
amostra_inferencia = df_check.sample(500, random_state=42).copy()

print("Iniciando a inferência de Sentimentos na amostra (500 frases)...")

# Função macete para aplicar no DataFrame e extrair só a tag
def prever_sentimento(frase):
    try:
        # truncation=True previne que o modelo "estoure" se topar com uma frase absurdamente longa
        resultado = sentiment_pipeline(frase, truncation=True, max_length=128)[0]
        return resultado['label'] # Retorna POS, NEG ou NEU
    except Exception as e:
        return "ERRO"

# Aplicando o modelo em cada frase com a nossa barra de progresso (versão texto)
amostra_inferencia['sentiment_label'] = amostra_inferencia['sentence'].progress_apply(prever_sentimento)

print("\nConcluído! Veja como ficou a nova coluna:")
# Visualizando as primeiras 20 frases com seus sentimentos classificados!
display(amostra_inferencia[['sentence', 'genre', 'sentiment_label']].head(20))


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Iniciando a inferência de Sentimentos na amostra (500 frases)...


100%|██████████| 500/500 [00:27<00:00, 17.89it/s]


Concluído! Veja como ficou a nova coluna:


,sentence,genre,sentiment_label
155273,"A condição da pista (tipo de solo, chuva, sol, etc.)",Racing,NEU
44548,70 das vezes q vou jogar é perfeito mesmo com gringos !.,Indie,POS
177539,"E não são feias simplesmente por serem feias, mas porque os designers do jogo simplesmente não conseguiram fazer uma arte gráfica bonita.",Sports,NEG
62269,"Bastante divertido e com um grande potencial para crescimento, por ainda estar em fase alfa conta com um grande número de bugs e mecânicas desbalanceadas, é preciso de um pouco de paciência.",RPG,POS
54549,Review para curadoria url grátis Brasilurl,Racing,NEU
88321,Viciante jogar em grupo.,Strategy,POS
152154,Incorporação de avaliação dos jogadores (exemplo a volta de apresentação ou aquecimento de pneus).,Racing,NEU
127454,"A maneira como ele caminha lentamente em direção a você, inflexível e sem emoção, é horripilante especialmente quando ele aparece de repente no final de um longo corredor.",Horror,NEG
31468,"O jogo consegue ser divertido até certo ponto (clock tower), onde se você esquecer completamente do RE3 clássico vai achar o game bacana.",Horror,POS
92603,"Se gosta de desafios ou simplesmente brincar de lego, pode comprar sem medo.",Strategy,NEU


In [8]:
# Garantindo que as frases não sejam "cortadas" (resumidas) pelo Pandas
pd.set_option('display.max_colwidth', None)

print("Exibindo as frases inteiras e seus sentimentos classificados:\n")
display(amostra_inferencia[['sentence', 'genre', 'sentiment_label']].head(20))

Exibindo as frases inteiras e seus sentimentos classificados:



,sentence,genre,sentiment_label
155273,"A condição da pista (tipo de solo, chuva, sol, etc.)",Racing,NEU
44548,70 das vezes q vou jogar é perfeito mesmo com gringos !.,Indie,POS
177539,"E não são feias simplesmente por serem feias, mas porque os designers do jogo simplesmente não conseguiram fazer uma arte gráfica bonita.",Sports,NEG
62269,"Bastante divertido e com um grande potencial para crescimento, por ainda estar em fase alfa conta com um grande número de bugs e mecânicas desbalanceadas, é preciso de um pouco de paciência.",RPG,POS
54549,Review para curadoria url grátis Brasilurl,Racing,NEU
88321,Viciante jogar em grupo.,Strategy,POS
152154,Incorporação de avaliação dos jogadores (exemplo a volta de apresentação ou aquecimento de pneus).,Racing,NEU
127454,"A maneira como ele caminha lentamente em direção a você, inflexível e sem emoção, é horripilante especialmente quando ele aparece de repente no final de um longo corredor.",Horror,NEG
31468,"O jogo consegue ser divertido até certo ponto (clock tower), onde se você esquecer completamente do RE3 clássico vai achar o game bacana.",Horror,POS
92603,"Se gosta de desafios ou simplesmente brincar de lego, pode comprar sem medo.",Strategy,NEU


In [1]:
# Verificando as colunas reais do arquivo processado na Fase 1
import pandas as pd

arquivo_processado = "data/processed/steam_reviews_sentences_COM_SENTIMENTO.parquet"
df_verificacao = pd.read_parquet(arquivo_processado)

print("Colunas do dataset:")
print(df_verificacao.columns.tolist())

print("\nPrimeiras linhas para conferência:")
display(df_verificacao.head(3))

Colunas do dataset:
['review', 'voted_up', 'votes_up', 'votes_funny', 'author.playtime_at_review', 'language', 'genre', 'review_clean', 'word_count', 'sentence', 'sentiment_label']

Primeiras linhas para conferência:


,review,voted_up,votes_up,votes_funny,author.playtime_at_review,language,genre,review_clean,word_count,sentence,sentiment_label
0,"Em minha opnião, o melhor post-apocalyptic da ...",True,5,0,2248.0,brazilian,Action,"Em minha opnião, o melhor postapocalyptic da a...",67,"Em minha opnião, o melhor postapocalyptic da a...",POS
1,"Joguei mais um bocado do Evolve, e até termine...",False,6,1,139.0,brazilian,Action,"Joguei mais um bocado do Evolve, e até termine...",367,"Joguei mais um bocado do Evolve, e até termine...",NEU
2,"Joguei mais um bocado do Evolve, e até termine...",False,6,1,139.0,brazilian,Action,"Joguei mais um bocado do Evolve, e até termine...",367,"Primeira coisa, é que é um jogo muito, muito m...",NEG
